<a href="https://colab.research.google.com/github/Jags-Kamani/ApacheSpark/blob/master/6_Delete_Duplicate_Emails.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Question:

Write a solution to delete all duplicate emails, keeping only one unique email with the smallest id.
For SQL users, please note that you are supposed to write a DELETE statement and not a SELECT one.
For Pandas users, please note that you are supposed to modify Person in place.
After running your script, the answer shown is the Person table. The driver will first compile and run your piece of code and then show the Person table. The final order of the Person table does not matter.


**Sample Input:**

Person table:



```
   id             email
0   1  john@example.com
1   2   bob@example.com
2   3  john@example.com

```



**Sample Output:**



```
   id             email
0   1  john@example.com
1   2   bob@example.com
```



**Your Answer here**

In [6]:
#SQL
DELETE FROM PERSON
WHERE id IN (
    SELECT id FROM (
        SELECT id, row_number() OVER (PARTITON BY email ORDER BY id) as rn
        FROM person
    ) as t
    WHERE rn > 1
)

#Pandas
import pandas as pd

data = {
    'id' : [1, 2, 3],
    'email' : ['john@example.com', 'bob@example.com', 'john@example.com']
}

person = pd.DataFrame(data)

def remove_duplicate(person : pd.DataFrame) -> None:
  person.sort_values(by='id', ascending=True, inplace=True)
  person.drop_duplicates(subset='email', keep='first', inplace=True)
  return person

remove_duplicate(person)


#PySpark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

df = spark.createDataFrame(
    [(1, "john@example.com"), (2, "bob@example.com"), (3, "john@example.com")],
    ["id", "email"]
)

# Window: partition by email, order by id (smallest first)
window_spec = Window.partitionBy("email").orderBy(col("id").asc())

# Assign row numbers
ranked_df = df.withColumn("row_number", row_number().over(window_spec))

# Keep only first occurrence (smallest id)
result = ranked_df.filter(col("row_number") == 1).drop("row_number")

result.show()



,id,email
0,1,john@example.com
1,2,bob@example.com


**Expected Output**



```
  id              email
0   1  test1@example.com
1   2  test2@example.com
3   4  test3@example.com
```

